In [4]:
# ============================================================
# NOTEBOOK 13 — BATCH 4 INVENTORY
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk


PROJECT_ROOT = Path(
    r"D:\Pancreatic_Cancer_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data"

RAW_CT_DIR = DATA_DIR / "raw_ct"

LABELS_DIR = DATA_DIR / "labels"

AUTO_LABEL_DIR = (
    LABELS_DIR / "Automatic_Labels"
)

MANUAL_LABEL_DIR = (
    LABELS_DIR / "Manual_Labels"
)

PROCESSED_DIR = DATA_DIR / "processed"

BATCH4_INVENTORY_FILE = (
    PROCESSED_DIR / "batch4_inventory.csv"
)

METADATA_FILE = (
    PROCESSED_DIR / "metadata.csv"
)

HELD_OUT_CASE = "100936_00001"


if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("=" * 70)
print("NOTEBOOK 13 — BATCH 4 INVENTORY")
print("=" * 70)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw CT directory:")
print(RAW_CT_DIR)

print("\nAutomatic labels:")
print(AUTO_LABEL_DIR)

print("\nManual labels:")
print(MANUAL_LABEL_DIR)

print("\nProcessed metadata:")
print(METADATA_FILE)

print("\nBatch 4 inventory:")
print(BATCH4_INVENTORY_FILE)

NOTEBOOK 13 — BATCH 4 INVENTORY
Project root:
D:\Pancreatic_Cancer_Thesis

Raw CT directory:
D:\Pancreatic_Cancer_Thesis\data\raw_ct

Automatic labels:
D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels

Manual labels:
D:\Pancreatic_Cancer_Thesis\data\labels\Manual_Labels

Processed metadata:
D:\Pancreatic_Cancer_Thesis\data\processed\metadata.csv

Batch 4 inventory:
D:\Pancreatic_Cancer_Thesis\data\processed\batch4_inventory.csv


In [5]:
# ============================================================
# CELL 2 — IDENTIFY BATCH 4 CT CASES
# ============================================================

print("=" * 70)
print("IDENTIFYING BATCH 4 CT CASES")
print("=" * 70)

ct_files = sorted(
    RAW_CT_DIR.glob("*_0000.nii.gz")
)

study_ids_all = sorted(
    p.name.replace("_0000.nii.gz", "")
    for p in ct_files
)

print(
    "Total CT files currently in raw_ct:",
    len(study_ids_all)
)

print(
    "\n100936_00001 present:",
    HELD_OUT_CASE in study_ids_all
)

batch4_candidate_ids = [
    study_id
    for study_id in study_ids_all
    if study_id != HELD_OUT_CASE
]

print(
    "Batch 4 candidates:",
    len(batch4_candidate_ids)
)

print("\nFirst 20:")
for study_id in batch4_candidate_ids[:20]:
    print(" ", study_id)

print("\nLast 20:")
for study_id in batch4_candidate_ids[-20:]:
    print(" ", study_id)

assert HELD_OUT_CASE not in batch4_candidate_ids

print("\n✓ Candidate list created.")

IDENTIFYING BATCH 4 CT CASES
Total CT files currently in raw_ct: 536

100936_00001 present: True
Batch 4 candidates: 535

First 20:
  101691_00001
  101692_00001
  101693_00001
  101694_00001
  101695_00001
  101696_00001
  101697_00001
  101698_00001
  101699_00001
  101700_00001
  101701_00001
  101702_00001
  101703_00001
  101704_00001
  101705_00001
  101706_00001
  101707_00001
  101708_00001
  101709_00001
  101710_00001

Last 20:
  102204_00001
  102205_00001
  102206_00001
  102207_00001
  102208_00001
  102209_00001
  102210_00001
  102211_00001
  102212_00001
  102213_00001
  102214_00001
  102215_00001
  102216_00001
  102217_00001
  102218_00001
  102219_00001
  102220_00001
  102221_00001
  102222_00001
  102223_00001

✓ Candidate list created.


In [6]:
# ============================================================
# CELL 3 — CHECK AGAINST EXISTING PROCESSED DATASET
# ============================================================

print("=" * 70)
print("CHECKING AGAINST EXISTING PROCESSED DATASET")
print("=" * 70)

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Metadata file not found:\n{METADATA_FILE}"
    )

metadata = pd.read_csv(
    METADATA_FILE
)

processed_ids = set(
    metadata["study_id"]
    .astype(str)
    .str.strip()
)

batch4_set = set(
    batch4_candidate_ids
)

already_processed = (
    batch4_set & processed_ids
)

new_batch4 = (
    batch4_set - processed_ids
)

print(
    "Batch 4 candidates      :",
    len(batch4_set)
)

print(
    "Already in metadata     :",
    len(already_processed)
)

print(
    "New relative to dataset:",
    len(new_batch4)
)

if already_processed:

    print("\nAlready processed:")
    for study_id in sorted(
        already_processed
    ):
        print(" ", study_id)

CHECKING AGAINST EXISTING PROCESSED DATASET
Batch 4 candidates      : 535
Already in metadata     : 0
New relative to dataset: 535


In [7]:
# ============================================================
# CELL 4 — CT FILE INVENTORY
# ============================================================

records = []

for study_id in batch4_candidate_ids:

    ct_path = (
        RAW_CT_DIR
        / f"{study_id}_0000.nii.gz"
    )

    records.append(
        {
            "study_id": study_id,
            "ct_exists": ct_path.exists(),
            "ct_path": str(ct_path),
            "ct_size_mb": (
                ct_path.stat().st_size
                / (1024 ** 2)
                if ct_path.exists()
                else np.nan
            ),
        }
    )

batch4_ct_inventory = pd.DataFrame(
    records
)

print("=" * 70)
print("BATCH 4 CT INVENTORY")
print("=" * 70)

print(
    "Cases:",
    len(batch4_ct_inventory)
)

print(
    "Missing CTs:",
    int(
        (~batch4_ct_inventory["ct_exists"])
        .sum()
    )
)

display(
    batch4_ct_inventory.head(20)
)

BATCH 4 CT INVENTORY
Cases: 535
Missing CTs: 0


,study_id,ct_exists,ct_path,ct_size_mb
0,101691_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101691...,210.165126
1,101692_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101692...,198.916186
2,101693_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101693...,139.378262
3,101694_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101694...,93.878595
4,101695_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101695...,210.935798
5,101696_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101696...,42.403634
6,101697_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101697...,24.356624
7,101698_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101698...,157.660740
8,101699_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101699...,47.018417
9,101700_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101700...,26.460115


In [8]:
# ============================================================
# CELL 5 — LABEL AVAILABILITY
# ============================================================

records = []

for study_id in batch4_candidate_ids:

    automatic_path = (
        AUTO_LABEL_DIR
        / f"{study_id}.nii.gz"
    )

    manual_path = (
        MANUAL_LABEL_DIR
        / f"{study_id}.nii.gz"
    )

    records.append(
        {
            "study_id": study_id,
            "ct_exists": (
                RAW_CT_DIR
                / f"{study_id}_0000.nii.gz"
            ).exists(),
            "automatic_label_exists":
                automatic_path.exists(),
            "manual_label_exists":
                manual_path.exists(),
        }
    )

batch4_inventory = pd.DataFrame(
    records
)

batch4_inventory["has_any_label"] = (
    batch4_inventory[
        "automatic_label_exists"
    ]
    |
    batch4_inventory[
        "manual_label_exists"
    ]
)

batch4_inventory["has_both_labels"] = (
    batch4_inventory[
        "automatic_label_exists"
    ]
    &
    batch4_inventory[
        "manual_label_exists"
    ]
)

batch4_inventory["has_automatic_only"] = (
    batch4_inventory[
        "automatic_label_exists"
    ]
    &
    ~batch4_inventory[
        "manual_label_exists"
    ]
)

batch4_inventory["has_manual_only"] = (
    batch4_inventory[
        "manual_label_exists"
    ]
    &
    ~batch4_inventory[
        "automatic_label_exists"
    ]
)

batch4_inventory["has_no_label"] = (
    ~batch4_inventory[
        "automatic_label_exists"
    ]
    &
    ~batch4_inventory[
        "manual_label_exists"
    ]
)


print("=" * 70)
print("BATCH 4 LABEL AVAILABILITY")
print("=" * 70)

print(
    "Total cases:",
    len(batch4_inventory)
)

print(
    "Automatic labels:",
    int(
        batch4_inventory[
            "automatic_label_exists"
        ].sum()
    )
)

print(
    "Manual labels:",
    int(
        batch4_inventory[
            "manual_label_exists"
        ].sum()
    )
)

print(
    "Both:",
    int(
        batch4_inventory[
            "has_both_labels"
        ].sum()
    )
)

print(
    "Automatic only:",
    int(
        batch4_inventory[
            "has_automatic_only"
        ].sum()
    )
)

print(
    "Manual only:",
    int(
        batch4_inventory[
            "has_manual_only"
        ].sum()
    )
)

print(
    "No label:",
    int(
        batch4_inventory[
            "has_no_label"
        ].sum()
    )
)

BATCH 4 LABEL AVAILABILITY
Total cases: 535
Automatic labels: 416
Manual labels: 119
Both: 0
Automatic only: 416
Manual only: 119
No label: 0


In [9]:
# ============================================================
# CELL 6 — SELECT LABEL TYPE
# ============================================================

def select_label(row):

    if row["automatic_label_exists"]:
        return "automatic"

    if row["manual_label_exists"]:
        return "manual"

    return None


batch4_inventory["selected_label_type"] = (
    batch4_inventory.apply(
        select_label,
        axis=1
    )
)

batch4_inventory["selected_label_path"] = None

for idx, row in batch4_inventory.iterrows():

    study_id = row["study_id"]

    if row["selected_label_type"] == "automatic":

        path = (
            AUTO_LABEL_DIR
            / f"{study_id}.nii.gz"
        )

    elif row["selected_label_type"] == "manual":

        path = (
            MANUAL_LABEL_DIR
            / f"{study_id}.nii.gz"
        )

    else:

        path = None

    batch4_inventory.loc[
        idx,
        "selected_label_path"
    ] = (
        str(path)
        if path is not None
        else None
    )


print("=" * 70)
print("SELECTED LABEL SUMMARY")
print("=" * 70)

print(
    batch4_inventory[
        "selected_label_type"
    ].value_counts(dropna=False)
)

SELECTED LABEL SUMMARY
selected_label_type
automatic    416
manual       119
Name: count, dtype: int64


In [10]:
# ============================================================
# CELL 7 — CT + LABEL READABILITY
# ============================================================

print("=" * 70)
print("BATCH 4 READABILITY CHECK")
print("=" * 70)

results = []

for i, row in enumerate(
    batch4_inventory.itertuples(
        index=False
    ),
    start=1,
):

    study_id = row.study_id

    ct_path = (
        RAW_CT_DIR
        / f"{study_id}_0000.nii.gz"
    )

    label_path = (
        Path(row.selected_label_path)
        if row.selected_label_path
        else None
    )

    result = {
        "study_id": study_id,
        "ct_readable": False,
        "label_readable": False,
        "ct_shape": None,
        "label_shape": None,
        "ct_spacing": None,
        "label_spacing": None,
        "ct_origin": None,
        "label_origin": None,
        "ct_direction": None,
        "label_direction": None,
        "error": None,
    }

    try:

        ct = sitk.ReadImage(
            str(ct_path)
        )

        result["ct_readable"] = True
        result["ct_shape"] = tuple(
            ct.GetSize()
        )
        result["ct_spacing"] = tuple(
            ct.GetSpacing()
        )
        result["ct_origin"] = tuple(
            ct.GetOrigin()
        )
        result["ct_direction"] = tuple(
            ct.GetDirection()
        )

        if label_path is None:
            raise FileNotFoundError(
                "No selected label."
            )

        label = sitk.ReadImage(
            str(label_path)
        )

        result["label_readable"] = True
        result["label_shape"] = tuple(
            label.GetSize()
        )
        result["label_spacing"] = tuple(
            label.GetSpacing()
        )
        result["label_origin"] = tuple(
            label.GetOrigin()
        )
        result["label_direction"] = tuple(
            label.GetDirection()
        )

    except Exception as e:

        result["error"] = str(e)

    results.append(result)

    if (
        i % 25 == 0
        or i == len(batch4_inventory)
    ):
        print(
            f"Checked {i}/{len(batch4_inventory)}"
        )


batch4_validation = pd.DataFrame(
    results
)

print("\n" + "-" * 70)

print(
    "CT readable:",
    int(
        batch4_validation[
            "ct_readable"
        ].sum()
    ),
    "/",
    len(batch4_validation),
)

print(
    "Label readable:",
    int(
        batch4_validation[
            "label_readable"
        ].sum()
    ),
    "/",
    len(batch4_validation),
)

BATCH 4 READABILITY CHECK
Checked 25/535
Checked 50/535
Checked 75/535
Checked 100/535
Checked 125/535
Checked 150/535
Checked 175/535
Checked 200/535
Checked 225/535
Checked 250/535
Checked 275/535
Checked 300/535
Checked 325/535
Checked 350/535
Checked 375/535
Checked 400/535
Checked 425/535
Checked 450/535
Checked 475/535
Checked 500/535
Checked 525/535
Checked 535/535

----------------------------------------------------------------------
CT readable: 535 / 535
Label readable: 535 / 535


In [11]:
# ============================================================
# CELL 8 — CT ↔ LABEL GEOMETRY
# ============================================================

SPACING_TOLERANCE = 1e-4
ORIGIN_TOLERANCE = 1e-4
DIRECTION_TOLERANCE = 1e-5

geometry_results = []

for _, row in batch4_validation.iterrows():

    result = {
        "study_id": row["study_id"],
        "shape_match": False,
        "spacing_match": False,
        "origin_match": False,
        "direction_match": False,
        "geometry_match": False,
        "error": row["error"],
    }

    if (
        not row["ct_readable"]
        or not row["label_readable"]
    ):
        geometry_results.append(result)
        continue

    result["shape_match"] = (
        row["ct_shape"]
        == row["label_shape"]
    )

    result["spacing_match"] = (
        np.allclose(
            row["ct_spacing"],
            row["label_spacing"],
            atol=SPACING_TOLERANCE,
        )
    )

    result["origin_match"] = (
        np.allclose(
            row["ct_origin"],
            row["label_origin"],
            atol=ORIGIN_TOLERANCE,
        )
    )

    result["direction_match"] = (
        np.allclose(
            row["ct_direction"],
            row["label_direction"],
            atol=DIRECTION_TOLERANCE,
        )
    )

    result["geometry_match"] = (
        result["shape_match"]
        and result["spacing_match"]
        and result["origin_match"]
        and result["direction_match"]
    )

    geometry_results.append(result)


batch4_geometry = pd.DataFrame(
    geometry_results
)


print("=" * 70)
print("BATCH 4 GEOMETRY VALIDATION")
print("=" * 70)

print(
    "Shape matching:",
    int(
        batch4_geometry[
            "shape_match"
        ].sum()
    ),
    "/",
    len(batch4_geometry),
)

print(
    "Spacing matching:",
    int(
        batch4_geometry[
            "spacing_match"
        ].sum()
    ),
    "/",
    len(batch4_geometry),
)

print(
    "Origin matching:",
    int(
        batch4_geometry[
            "origin_match"
        ].sum()
    ),
    "/",
    len(batch4_geometry),
)

print(
    "Direction matching:",
    int(
        batch4_geometry[
            "direction_match"
        ].sum()
    ),
    "/",
    len(batch4_geometry),
)

print(
    "Full geometry matching:",
    int(
        batch4_geometry[
            "geometry_match"
        ].sum()
    ),
    "/",
    len(batch4_geometry),
)

BATCH 4 GEOMETRY VALIDATION
Shape matching: 535 / 535
Spacing matching: 535 / 535
Origin matching: 535 / 535
Direction matching: 535 / 535
Full geometry matching: 535 / 535


In [12]:
# ============================================================
# CELL 9 — BUILD FINAL BATCH 4 INVENTORY
# ============================================================

batch4_inventory = batch4_inventory.merge(
    batch4_validation,
    on="study_id",
    how="left",
)

batch4_inventory = batch4_inventory.merge(
    batch4_geometry[
        [
            "study_id",
            "shape_match",
            "spacing_match",
            "origin_match",
            "direction_match",
            "geometry_match",
        ]
    ],
    on="study_id",
    how="left",
)


print("=" * 70)
print("FINAL BATCH 4 INVENTORY TABLE")
print("=" * 70)

print(
    "Shape:",
    batch4_inventory.shape
)

display(
    batch4_inventory.head()
)

FINAL BATCH 4 INVENTORY TABLE
Shape: (535, 27)


,study_id,ct_exists,automatic_label_exists,manual_label_exists,has_any_label,has_both_labels,has_automatic_only,has_manual_only,has_no_label,selected_label_type,...,ct_origin,label_origin,ct_direction,label_direction,error,shape_match,spacing_match,origin_match,direction_match,geometry_match
0,101691_00001,True,True,False,True,False,True,False,False,automatic,...,"(-273.79998779296875, -250.0, -519.875)","(-273.79998779296875, -250.0, -519.875)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
1,101692_00001,True,True,False,True,False,True,False,False,automatic,...,"(-152.0760040283203, -88.30000305175781, -7.55...","(-152.0760040283203, -88.30000305175781, -7.55...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
2,101693_00001,True,True,False,True,False,True,False,False,automatic,...,"(-158.89999389648438, -45.400001525878906, -3....","(-158.89999389648438, -45.400001525878906, -3....","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
3,101694_00001,True,False,True,True,False,False,True,False,manual,...,"(-164.1552734375, -338.6552734375, 876.9000244...","(-164.1552734375, -338.6552734375, 876.9000244...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
4,101695_00001,True,False,True,True,False,False,True,False,manual,...,"(-182.625, -55.599998474121094, 33.09999847412...","(-182.625, -55.599998474121094, 33.09999847412...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True


In [13]:
# ============================================================
# CELL 10 — CASES REQUIRING REVIEW
# ============================================================

problems = batch4_inventory[
    (~batch4_inventory["ct_readable"])
    | (~batch4_inventory["label_readable"])
    | (~batch4_inventory["geometry_match"])
].copy()

print("=" * 70)
print("BATCH 4 CASES REQUIRING REVIEW")
print("=" * 70)

print(
    "Cases requiring review:",
    len(problems)
)

if len(problems) > 0:

    display(
        problems[
            [
                "study_id",
                "automatic_label_exists",
                "manual_label_exists",
                "selected_label_type",
                "ct_readable",
                "label_readable",
                "shape_match",
                "spacing_match",
                "origin_match",
                "direction_match",
                "geometry_match",
                "error",
            ]
        ]
    )

else:

    print(
        "✓ No readability or geometry problems."
    )

BATCH 4 CASES REQUIRING REVIEW
Cases requiring review: 0
✓ No readability or geometry problems.


In [14]:
# ============================================================
# CELL 11 — SAVE BATCH 4 INVENTORY
# ============================================================

BATCH4_INVENTORY_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

batch4_inventory.to_csv(
    BATCH4_INVENTORY_FILE,
    index=False,
)

print("=" * 70)
print("BATCH 4 INVENTORY SAVED")
print("=" * 70)

print(
    "Path:",
    BATCH4_INVENTORY_FILE
)

print(
    "Rows:",
    len(batch4_inventory)
)

BATCH 4 INVENTORY SAVED
Path: D:\Pancreatic_Cancer_Thesis\data\processed\batch4_inventory.csv
Rows: 535


In [15]:
# ============================================================
# CELL 12 — FINAL BATCH 4 INVENTORY REPORT
# ============================================================

print("=" * 70)
print("FINAL BATCH 4 INVENTORY REPORT")
print("=" * 70)

total = len(batch4_inventory)

print(
    "Batch 4 CT candidates :",
    total
)

print(
    "CT readable           :",
    int(
        batch4_inventory[
            "ct_readable"
        ].sum()
    ),
    "/",
    total,
)

print(
    "Label readable        :",
    int(
        batch4_inventory[
            "label_readable"
        ].sum()
    ),
    "/",
    total,
)

print(
    "Geometry matching     :",
    int(
        batch4_inventory[
            "geometry_match"
        ].sum()
    ),
    "/",
    total,
)

print("\nSelected labels:")
print(
    batch4_inventory[
        "selected_label_type"
    ].value_counts(
        dropna=False
    )
)

print("\nCases requiring review:")
print(len(problems))

print("=" * 70)

FINAL BATCH 4 INVENTORY REPORT
Batch 4 CT candidates : 535
CT readable           : 535 / 535
Label readable        : 535 / 535
Geometry matching     : 535 / 535

Selected labels:
selected_label_type
automatic    416
manual       119
Name: count, dtype: int64

Cases requiring review:
0
